## Importing Packages

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor

import mlflow
import mlflow.sklearn
import mlflow.xgboost

## Load teh Data\n\n`load_boston` was removed from scikit-learn (ethical concerns with one of the original features), so we pull the same classic dataset from a public CSV mirror. Target `medv` = median home value in $1000s.

In [ ]:
url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
df = pd.read_csv(url)

X = df.drop(columns=["medv"])
y = df["medv"]

In [ ]:
print("Shape:", X.shape)
print("Features:", list(X.columns))
print("Target stats:")
print(y.describe())

## Train Ready

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

## Diff Exp Models

In [ ]:
lin_reg = LinearRegression()

lin_reg.fit(X_train, y_train)

y_pred = lin_reg.predict(X_test)

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("R2  :", r2_score(y_test, y_pred))

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("R2  :", r2_score(y_test, y_pred))

In [ ]:
xgb = XGBRegressor(
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("R2  :", r2_score(y_test, y_pred))

In [ ]:
xgb_tuned = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

xgb_tuned.fit(
    X_train,
    y_train
)

y_pred = xgb_tuned.predict(X_test)

print("Tuned XGBoost")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("R2  :", r2_score(y_test, y_pred))

## Prepare for ML Flow

In [ ]:
models = [

    (
        "Linear Regression",
        LinearRegression(),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "XGBoost",
        XGBRegressor(
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "XGBoost (Tuned)",
        XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    )

]

In [ ]:
reports = []

for model_name, model, train_set, test_set in models:

    X_tr, y_tr = train_set
    X_te, y_te = test_set

    model.fit(X_tr, y_tr)

    predictions = model.predict(X_te)

    report = {
        "rmse": np.sqrt(mean_squared_error(y_te, predictions)),
        "mae": mean_absolute_error(y_te, predictions),
        "r2": r2_score(y_te, predictions)
    }

    reports.append(report)

In [ ]:
reports

## Exp Tracking ML Flow Local

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Boston Housing Regression PBLM 1")

In [ ]:
for i, (model_name, model, _, _) in enumerate(models):

    report = reports[i]

    with mlflow.start_run(run_name=model_name):
        #Params
        mlflow.log_param("Model", model_name)

        if hasattr(model, "get_params"):
            mlflow.log_params(model.get_params())

        #Mts
        mlflow.log_metric(
            "RMSE",
            report["rmse"]
        )

        mlflow.log_metric(
            "MAE",
            report["mae"]
        )

        mlflow.log_metric(
            "R2",
            report["r2"]
        )

        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(
                model,
                artifact_path="model"
            )
        else:
            mlflow.sklearn.log_model(
                model,
                artifact_path="model"
            )

## Reg teh Best Model

In [ ]:
best_index = np.argmax(
    [report["r2"] for report in reports]
)


best_model_name, best_model, _, _ = models[best_index]

best_report = reports[best_index]

print("Model:", best_model_name)
print("RMSE:", best_report["rmse"])
print("R2  :", best_report["r2"])

In [ ]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:
    #Params
    mlflow.log_param(
        "Model",
        best_model_name
    )
    mlflow.log_param(
        "Selection_Metric",
        "R2 Score"
    )
    mlflow.log_param(
        "Dataset",
        "Boston Housing Dataset"
    )
    mlflow.log_params(
        best_model.get_params()
    )
    #Metrics
    mlflow.log_metric(
        "RMSE",
        best_report["rmse"]
    )

    mlflow.log_metric(
        "MAE",
        best_report["mae"]
    )

    mlflow.log_metric(
        "R2",
        best_report["r2"]
    )

    mlflow.set_tag(
        "Model_Type",
        best_model_name
    )

    mlflow.set_tag(
        "Stage",
        "Candidate"
    )

    mlflow.set_tag(
        "Task",
        "Regression"
    )

    if "XGBoost" in best_model_name:

        model_info = mlflow.xgboost.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Boston_Housing_Best_Model"
        )

    else:

        model_info = mlflow.sklearn.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Boston_Housing_Best_Model"
        )

    run_id = run.info.run_id
    model_uri = model_info.model_uri

print("Run ID       :", run_id)
print("Model URI    :", model_uri)
print("Model Name   :", "Boston_Housing_Best_Model")

## Loading and Pushing to Prod

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

latest_version = client.get_latest_versions(
    "Boston_Housing_Best_Model"
)[0].version

print(latest_version)

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient


mlflow.set_tracking_uri(
    "http://127.0.0.1:5000"
)


client = MlflowClient()

In [ ]:
model_name = "Boston_Housing_Best_Model"


latest_version = client.get_latest_versions(
    model_name
)[0]


print("Model Name:", latest_version.name)
print("Version:", latest_version.version)
print("Stage:", latest_version.current_stage)

In [ ]:
model_uri = f"models:/{model_name}/{latest_version.version}"


model = mlflow.pyfunc.load_model(
    model_uri
)


print("Model loaded successfully")

In [ ]:
predictions = model.predict(
    X_test
)


print(predictions[:10])

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))
print("R2  :", r2_score(y_test, predictions))

In [ ]:
client.update_model_version(
    name=model_name,
    version=latest_version.version,
    description="""
    Champion model for Boston Housing Regression.

    Tested successfully before production deployment.

    Dataset:
    Boston Housing Dataset

    Metric:
    R2 Score
    """
)

In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=latest_version.version,
    stage="Production"
)

In [ ]:
production_model = mlflow.pyfunc.load_model(
    "models:/Boston_Housing_Best_Model/Production"
)
prediction = production_model.predict(
    X_test
)
print(prediction[:10])

In [ ]:
production_versions = client.get_latest_versions(
    "Boston_Housing_Best_Model",
    stages=["Production"]
)
for model in production_versions:
    print(
        "Version:",
        model.version
    )
    print(
        "Run ID:",
        model.run_id
    )
    print(
        "Stage:",
        model.current_stage
    )